# SBP calibration

Map each sub-thread's two measured XYZ endpoints and motor step counts to bead positions. Each sub-thread gets its own steps-per-metre conversion. The calibration also records the margins to skip at its ends.

Run from the repository root. The default is **simulation**, using example endpoints and step counts. For a real calibration, enter the measured XYZ coordinates, set the serial connection and travel limits in your local configuration, then capture each endpoint. The rig has no automatic hardware home; home and position refer to a manually established reference.

In [ ]:
from pathlib import Path
from src.calibration import Calibration, capture_endpoint, make_calibration
from src.acquisition import MeasurementConfig, build_motor, create_demo_config
from src.motor import jog_background, show_emergency_stop, list_serial_ports

SIMULATE = True
CONFIG_FILE = (create_demo_config("outputs/calibration") if SIMULATE
               else Path("config/measurement_config.json"))
config = MeasurementConfig.from_file(CONFIG_FILE)

## 1. Enter the sub-thread endpoints

XYZ values below are **examples**. Replace them for your setup before taking a real calibration. Coordinates may use your chosen booster frame; they are not assumed to be distances from the mirror.

`XYZ_UNIT_M=1` means metres; use `0.01` for centimetres or `0.001` for millimetres. Margins are always in metres. The Euclidean endpoint separation determines the scale, so the coordinate unit matters.

In [ ]:
XYZ_UNIT_M = 1.0
SUB_THREADS = [
    {"index": 0, "name": "T0", "start_xyz": [-0.1, 0, 0], "end_xyz": [0.1, 0, 0]},
    {"index": 1, "name": "T1", "start_xyz": [-0.1, 0, 0.02], "end_xyz": [0.1, 0, 0.02],
     "margin_start_m": 0.01, "margin_end_m": 0.01},
    {"index": 2, "name": "T2", "start_xyz": [-0.1, 0, 0.04], "end_xyz": [0.1, 0, 0.04]},
    {"index": 3, "name": "T3", "start_xyz": [-0.1, 0, 0.06], "end_xyz": [0.1, 0, 0.06],
     "margin_start_m": 0.01, "margin_end_m": 0.01},
]

## 2. Connect and set travel limits

For a real motor, install the `hardware` dependencies and set `stepper.port` in `config/measurement_config.json`. `list_serial_ports()` lists available devices without opening them.

Both motors enforce the configured minimum and maximum step counts. The red STOP button works during `jog_background` moves. After a real emergency stop, re-establish the bead's position with `motor.set_position(known_steps)` before moving again.

In [ ]:
# list_serial_ports()
motor = build_motor(config, simulate=SIMULATE)
motor.set_limits(config.stepper.min_position_steps, config.stepper.max_position_steps)
print(motor.limits_description())
print("Current position:", motor.get_position(), "steps")
stop_button = show_emergency_stop(motor)

To establish new limits, jog to the endpoints and use `motor.set_min_position()` / `motor.set_max_position()`, then `motor.save_limits(CONFIG_FILE)`. The `motor.bypass_limits()` context is available when deliberately establishing the range; it re-arms the check on exit.

Set a repeatable home while the bead is physically at that reference. The position and home of the real motor share the original state file, `~/.madmax_bead_pull/motor_state.json`, unless `stepper.state_file` overrides it. Simulation is in memory.

In [ ]:
# For a real motor, declare a known position only while the bead is at that reference:
# motor.set_position(known_steps)
# motor.set_home()
print("Stored home:", motor.home_position)

## 3. Capture the start and end of each sub-thread

In simulation the following cell runs a short example for every sub-thread. For a real calibration, use the manual cells below and repeat them for every index. Jog commands are commented until you choose a movement.

In [ ]:
captured = {}
if SIMULATE:
    example = Calibration.from_calibration_file("config/bead_pull_calibration.example.json")
    for sub in example.sub_threads:
        motor.move_to_position(sub.start_steps)
        capture_endpoint(motor, captured, sub.index, "start")
        motor.move_to_position(sub.end_steps)
        capture_endpoint(motor, captured, sub.index, "end")

In [ ]:
THREAD_INDEX = 0
# move = jog_background(motor, 200)
# move.join()  # wait before starting another movement
# capture_endpoint(motor, captured, THREAD_INDEX, "start")
# Jog to the other endpoint, then:
# capture_endpoint(motor, captured, THREAD_INDEX, "end")
captured

## 4. Check and save

All entered sub-threads must have distinct start/end step counts and a nonzero physical length. The calibration file stores XYZ coordinates in metres and retains the original schema (version 2). Inspect each scale before saving.

In [ ]:
calibration = make_calibration(SUB_THREADS, captured, coordinate_unit_m=XYZ_UNIT_M)
print(calibration.summary())
calibration.save(config.calibration_file)
reloaded = Calibration.from_calibration_file(config.calibration_file)
print("Saved:", config.calibration_file)

## 5. Disconnect

For a real calibration, run this cell when finished, including after interrupted work.

In [ ]:
motor.shutdown()